<a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_10/07_text_data_augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://kaggle.com/kernels/welcome?src=https://github.com/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_10/07_text_data_augmentation.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>
<a href="https://github.com/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_10/07_text_data_augmentation.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github" alt="View Source On GitHub"/></a>

# Retrieval-based Text Data Augmentation mit LLMs

Dieses Notebook zeigt, wie man Textdaten mithilfe eines Large Language Models (LLM) augmentiert. Dabei nutzen wir einen Retrieval-basierten Ansatz: Wir geben dem LLM einige Beispiele einer bestimmten Klasse aus unserem Datensatz als Kontext (Few-Shot Prompting), damit es neue, ähnliche Beispiele generieren kann.

In [ ]:
# Installation notwendiger Bibliotheken
!pip install llm-client scikit-learn

In [ ]:
import numpy as np
import random
from sklearn.datasets import fetch_20newsgroups
from llm_client import LLMClient
import os

## Datensatz laden

Wir laden den 20 Newsgroups Datensatz. Dieser enthält ca. 11.000 Newsgroup-Posts (Trainings-Set), die in 20 Themengebiete unterteilt sind.

In [ ]:
# Datensatz laden (Training Subset)
newsgroups = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
class_names = newsgroups.target_names

print(f"Anzahl der Dokumente: {len(newsgroups.data)}")
print(f"Anzahl der Klassen: {len(class_names)}")

### Klassenstatistik

Hier sehen wir, wie viele Beispiele pro Klasse im Datensatz vorhanden sind. Dies hilft bei der Entscheidung, für welche Klasse eine Augmentierung sinnvoll sein könnte.

In [ ]:
targets, counts = np.unique(newsgroups.target, return_counts=True)
for target, count in zip(targets, counts):
    print(f"{class_names[target]:<30}: {count} Beispiele")

## Konfiguration der Augmentierung

Bitte wählen Sie die Zielklasse, die Anzahl der Kontext-Beispiele (N) und die Anzahl der zu generierenden Beispiele.

In [ ]:
#@title Augmentierungseinstellungen
target_class_name = "alt.atheism" #@param ["alt.atheism", "comp.graphics", "comp.os.ms-windows.misc", "comp.sys.ibm.pc.hardware", "comp.sys.mac.hardware", "comp.windows.x", "misc.forsale", "rec.autos", "rec.motorcycles", "rec.sport.baseball", "rec.sport.hockey", "sci.crypt", "sci.electronics", "sci.med", "sci.space", "soc.religion.christian", "talk.politics.guns", "talk.politics.mideast", "talk.politics.misc", "talk.religion.misc"]
n_context_examples = 2 #@param {type:"slider", min:1, max:10, step:1}
n_to_generate = 1 #@param {type:"integer"}

# Hinweis: Um ALLE Beispiele der gewählten Klasse als Kontext zu nutzen (Vorsicht bei Context-Window des LLMs!), 
# kann man n_context_examples auf die Anzahl der Dokumente in der Klasse setzen:
# target_class_idx = class_names.index(target_class_name)
# class_indices = [i for i, t in enumerate(newsgroups.target) if t == target_class_idx]
# n_context_examples = len(class_indices)

## Retrieval und Prompt-Erstellung

Wir rufen zufällig $N$ Beispiele der gewählten Klasse ab und erstellen daraus einen Prompt für das LLM.

In [ ]:
# Index der Zielklasse finden
target_class_idx = class_names.index(target_class_name)

# Alle Indizes dieser Klasse finden
class_indices = [i for i, t in enumerate(newsgroups.target) if t == target_class_idx]

# Zufällige Auswahl von N Beispielen
selected_indices = random.sample(class_indices, min(n_context_examples, len(class_indices)))
context_examples = [newsgroups.data[i] for i in selected_indices]

# Prompt zusammenstellen
prompt = f"Du bist ein Experte für Textgenerierung. Hier sind {len(context_examples)} Beispiele für Texte aus der Kategorie '{target_class_name}':\n\n"

for i, text in enumerate(context_examples):
    prompt += f"--- Beispiel {i+1} ---\n{text.strip()}\n\n"

prompt += f"Bitte generiere {n_to_generate} weitere, neue und realistische Beispiele für Texte aus der Kategorie '{target_class_name}'. "
prompt += "Die generierten Texte sollten in Stil und Inhalt den obigen Beispielen ähneln, aber eigenständige Inhalte haben. "
prompt += "Trenne die Beispiele durch '--- GENERIERTES BEISPIEL ---'."

# Prompt in Datei speichern
with open("augmentation_prompt.md", "w") as f:
    f.write(prompt)

print("Prompt wurde erstellt und in 'augmentation_prompt.md' gespeichert.")

## LLM Aufruf und Ergebnis

Wir nutzen den `llm_client`, um die Beispiele zu generieren.

In [ ]:
# LLM Client initialisieren (nutzt API-Keys aus Umgebungsvariablen oder Colab-Secrets)
client = LLMClient()

messages = [
    {"role": "system", "content": "Du bist ein hilfreicher KI-Assistent für die Datenaugmentierung."},
    {"role": "user", "content": prompt}
]

print("Generiere Beispiele... Dies kann einen Moment dauern.")
response = client.chat_completion(messages)

print("\n--- GENERIERTE ERGEBNISSE ---")
print(response)

## Vergleich mit Originaldaten

Hier wird eines der generierten Beispiele einem Beispiel aus dem ursprünglichen Trainingsdatensatz gegenübergestellt.

In [ ]:
print(f"\n=== ORIGINALBEISPIEL DER KLASSE {target_class_name} ===")
# Wir zeigen ein zufälliges Beispiel aus den für den Prompt genutzten
print(context_examples[0].strip())

print(f"\n=== GENERIERTE(S) BEISPIEL(E) ===")
print(response.strip())